Evaluation of Multilingual Pretraining for Low-Resource Neural Machine Translation
English→Yoruba and English→Khmer (OPUS-100)


## 1. Environment setup

In [ ]:
!pip install -q -U "transformers>=4.41" "datasets>=2.19,<3.0" sentencepiece sacrebleu evaluate accelerate sacremoses seaborn
import transformers
print("transformers", transformers.__version__)


In [3]:
import os, json, random, time, math, gc, itertools
from dataclasses import dataclass, field
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset

from transformers import (
    AutoTokenizer, AutoModel, EncoderDecoderModel,
    MT5ForConditionalGeneration,
    Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq,
    get_inverse_sqrt_schedule
)
import sacrebleu
from scipy import stats

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

Device: cuda


## 2. Experiment configuration

In [ ]:
FULL_SCALE = False   # True => full-scale splits/steps (needs a strong GPU + hours)

LANG_PAIRS = {
    # NOTE: OPUS-100 (Helsinki-NLP/opus-100) has no English-Swahili configuration, so this
    # study covers English-Yoruba and English-Khmer only.
    "en-yo": {"hf_config": "en-yo", "name": "English->Yoruba"},
    "en-km": {"hf_config": "en-km", "name": "English->Khmer"},
}

CPU_FAST_MODE = not torch.cuda.is_available()

if FULL_SCALE:
    N_TRAIN, N_VAL, N_TEST = 8000, 1000, 1000
    MAX_STEPS = 100_000
    EVAL_EVERY = 2000
    PATIENCE = 5
    SEEDS = [13, 42, 123, 777, 2024]
    BPE_VOCAB = 32000
elif CPU_FAST_MODE:
    N_TRAIN, N_VAL, N_TEST = 500, 100, 100
    MAX_STEPS = 500
    EVAL_EVERY = 100
    PATIENCE = 3
    SEEDS = [13]              # one seed only -- bump back up once you have a GPU
    BPE_VOCAB = 2000
else:
    # Scaled-down defaults so the full pipeline completes on a free Colab GPU.
    N_TRAIN, N_VAL, N_TEST = 3000, 400, 400
    MAX_STEPS = 3000
    EVAL_EVERY = 300
    PATIENCE = 3
    SEEDS = [13, 42]          # use 5 seeds for tighter confidence intervals
    BPE_VOCAB = 8000

# All four models are enabled now that GPU compute is available (mBERT/mT5 were
# previously infeasible on CPU: 46+ hour / 300+ hour ETAs observed there).
MODEL_NAMES = ["LSTM-Attention", "Transformer", "Transformer+Pretraining", "mT5-Style"]
MODEL_NAMES_TO_RUN = MODEL_NAMES
MAX_LEN = 64
BATCH_SIZE = 32

# Set to True (and run this cell in Colab) to save results/checkpoints to Google Drive
# instead of the local VM disk. This matters because a Colab disconnect or timeout can
# recycle the VM entirely, wiping ./data and ./results along with it -- Drive survives that. Highly recommended for FULL_SCALE runs (multi-hour).
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/nmt_experiment/data"
    RESULTS_DIR = "/content/drive/MyDrive/nmt_experiment/results"
else:
    DATA_DIR = "./data"
    RESULTS_DIR = "./results"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Scale: {'FULL' if FULL_SCALE else 'REDUCED'} | train={N_TRAIN} val={N_VAL} test={N_TEST} | seeds={SEEDS}")

## 3. Real parallel data (OPUS-100)

We use **OPUS-100** (Zhang et al., 2020; Helsinki-NLP), a publicly available English-centric
multilingual corpus mined from OPUS that contains En-Yo and En-Km parallel data and is directly
loadable from Hugging Face.

Splits are drawn deterministically (fixed seed) from OPUS-100's train/validation/test partitions,
then truncated to `N_TRAIN/N_VAL/N_TEST` so that every language pair has the same fixed training budget.

In [5]:
import unicodedata, re

def normalize(text):
    text = unicodedata.normalize("NFKC", text)
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text

def load_pair(pair_key, cfg):
    actual_hf_config_name = cfg["hf_config"]

    ds_dict = load_dataset("Helsinki-NLP/opus-100", actual_hf_config_name)

    # Ensure 'train' split exists, as it's the base for creating others if needed
    if 'train' not in ds_dict:
        raise ValueError(f"No 'train' split found for config '{actual_hf_config_name}'. Cannot proceed.")

    # If 'validation' split is missing, create it from the 'train' split
    if 'validation' not in ds_dict:
        print(f"Warning: 'validation' split not found for config '{actual_hf_config_name}'. Creating a validation split from the training data.")

        # Check if there's enough data in current 'train' to extract N_VAL samples
        if len(ds_dict['train']) >= N_VAL:
            train_val_split = ds_dict['train'].train_test_split(test_size=N_VAL, seed=42)
            ds_dict['train'] = train_val_split['train']
            ds_dict['validation'] = train_val_split['test']
        else:
            print(f"Warning: 'train' split too small to create a separate validation set of size {N_VAL}. Assigning an empty Dataset to 'validation'.")
            ds_dict['validation'] = ds_dict['train'].select(range(0)) # Empty Dataset

    # If 'test' split is missing, create it from the remaining 'train' split
    # Note: 'train' might have been modified by the validation split creation above.
    if 'test' not in ds_dict:
        print(f"Warning: 'test' split not found for config '{actual_hf_config_name}'. Creating a test split from the training data.")

        # Check if there's enough data in current 'train' to extract N_TEST samples
        if len(ds_dict['train']) >= N_TEST:
            train_test_split = ds_dict['train'].train_test_split(test_size=N_TEST, seed=42)
            ds_dict['train'] = train_test_split['train']
            ds_dict['test'] = train_test_split['test']
        else:
            print(f"Warning: 'train' split too small to create a separate test set of size {N_TEST}. Assigning an empty Dataset to 'test'.")
            ds_dict['test'] = ds_dict['train'].select(range(0)) # Empty Dataset


    tgt_lang = pair_key.split("-")[1]

    def to_rows(split_name, n_samples):
        # Access the dataset split by name from the potentially modified ds_dict
        dataset_split = ds_dict[split_name]

        rows = dataset_split.shuffle(seed=42).select(range(min(n_samples, len(dataset_split))))
        src_texts, tgt_texts = [], []
        for ex in rows["translation"]:
            # Ensure 'en' is always the source and 'tgt_lang' is always the target
            s = normalize(ex["en"])
            t = normalize(ex[tgt_lang])
            if len(s) == 0 or len(t) == 0:
                continue
            src_texts.append(s); tgt_texts.append(t)
        return src_texts, tgt_texts

    train_src, train_tgt = to_rows("train", N_TRAIN)
    val_src, val_tgt = to_rows("validation", N_VAL)
    test_src, test_tgt = to_rows("test", N_TEST)
    return {
        "train": (train_src, train_tgt),
        "val": (val_src, val_tgt),
        "test": (test_src, test_tgt),
    }

DATA = {}
for pk, cfg in LANG_PAIRS.items():
    print(f"Loading {pk} ({cfg['name']}) from OPUS-100 ...")
    DATA[pk] = load_pair(pk, cfg)
    tr, va, te = DATA[pk]["train"], DATA[pk]["val"], DATA[pk]["test"]
    print(f"  train={len(tr[0])}  val={len(va[0])}  test={len(te[0])}")

# quick sanity peek
for pk in LANG_PAIRS:
    s, t = DATA[pk]["train"][0][0], DATA[pk]["train"][1][0]
    print(pk, "|", s, "->", t)

Loading en-yo (English->Yoruba) from OPUS-100 ...


Generating train split:   0%|          | 0/10375 [00:00<?, ? examples/s]

  train=3000  val=400  test=400
Loading en-km (English->Khmer) from OPUS-100 ...


Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/111483 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

  train=3000  val=400  test=400
en-yo | english (_us-ascii) -> gẹ́ẹ̀sì(_us-ascii)
en-km | sets the default button color -> កំណត់​ពណ៌​ប៊ូតុង​លំនាំដើម​


## Data-quality audit (contamination rate, train–test near-duplicate overlap, short-segment proportion)

In [ ]:

def _char_ngrams(s, n=3):
    return set(s[i:i+n] for i in range(max(0, len(s) - n + 1)))

def _jaccard(a, b):
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

def audit_pair(pair_key, near_dup_threshold=0.7):
    src_tr, tgt_tr = DATA[pair_key]["train"]
    src_te, tgt_te = DATA[pair_key]["test"]
    n_test = len(src_te)

    train_src_set = set(src_tr)
    train_pair_set = set(zip(src_tr, tgt_tr))

    exact_src_dup = sum(1 for s in src_te if s in train_src_set)
    exact_pair_dup = sum(1 for p in zip(src_te, tgt_te) if p in train_pair_set)

    train_ngrams = [_char_ngrams(s) for s in src_tr]
    near_dup_count = 0
    for s in src_te:
        te_ng = _char_ngrams(s)
        if any(_jaccard(te_ng, tn) >= near_dup_threshold for tn in train_ngrams):
            near_dup_count += 1

    def short_frac(texts, max_tok=5):
        # NOTE: whitespace-based token count. Meaningful for space-delimited scripts
        # (e.g. Yoruba); undercounts for scripts without whitespace word boundaries
        # (e.g. Khmer) -- interpret the Khmer row as a rough proxy, not a precise count.
        return 100 * sum(1 for t in texts if len(t.split()) <= max_tok) / len(texts) if texts else 0.0

    return {
        "pair": pair_key,
        "n_test": n_test,
        "exact_src_duplicate_pct": round(100 * exact_src_dup / n_test, 2) if n_test else 0.0,
        "exact_pair_duplicate_pct (contamination rate)": round(100 * exact_pair_dup / n_test, 2) if n_test else 0.0,
        "near_duplicate_src_pct (jaccard>=0.7)": round(100 * near_dup_count / n_test, 2) if n_test else 0.0,
        "pct_test_src_le5_tokens": round(short_frac(src_te), 2),
        "pct_test_tgt_le5_tokens": round(short_frac(tgt_te), 2),
    }

audit_rows = [audit_pair(pk) for pk in LANG_PAIRS]
audit_df = pd.DataFrame(audit_rows)
print("Data-quality audit (contamination / near-duplicate / short-segment rates)")
display(audit_df)
audit_df.to_csv(f"{RESULTS_DIR}/table_data_quality_audit.csv", index=False)


In [6]:
from datasets import get_dataset_config_names

print("Available configurations for Helsinki-NLP/opus-100:")
print(get_dataset_config_names("Helsinki-NLP/opus-100"))

Available configurations for Helsinki-NLP/opus-100:
['af-en', 'am-en', 'an-en', 'ar-de', 'ar-en', 'ar-fr', 'ar-nl', 'ar-ru', 'ar-zh', 'as-en', 'az-en', 'be-en', 'bg-en', 'bn-en', 'br-en', 'bs-en', 'ca-en', 'cs-en', 'cy-en', 'da-en', 'de-en', 'de-fr', 'de-nl', 'de-ru', 'de-zh', 'dz-en', 'el-en', 'en-eo', 'en-es', 'en-et', 'en-eu', 'en-fa', 'en-fi', 'en-fr', 'en-fy', 'en-ga', 'en-gd', 'en-gl', 'en-gu', 'en-ha', 'en-he', 'en-hi', 'en-hr', 'en-hu', 'en-hy', 'en-id', 'en-ig', 'en-is', 'en-it', 'en-ja', 'en-ka', 'en-kk', 'en-km', 'en-kn', 'en-ko', 'en-ku', 'en-ky', 'en-li', 'en-lt', 'en-lv', 'en-mg', 'en-mk', 'en-ml', 'en-mn', 'en-mr', 'en-ms', 'en-mt', 'en-my', 'en-nb', 'en-ne', 'en-nl', 'en-nn', 'en-no', 'en-oc', 'en-or', 'en-pa', 'en-pl', 'en-ps', 'en-pt', 'en-ro', 'en-ru', 'en-rw', 'en-se', 'en-sh', 'en-si', 'en-sk', 'en-sl', 'en-sq', 'en-sr', 'en-sv', 'en-ta', 'en-te', 'en-tg', 'en-th', 'en-tk', 'en-tr', 'en-tt', 'en-ug', 'en-uk', 'en-ur', 'en-uz', 'en-vi', 'en-wa', 'en-xh', 'en-yi', 'e

## Target-language-identification audit (extends the data-quality audit above)

In [ ]:

def khmer_script_fraction(text):
    if not text:
        return 0.0
    khmer_chars = sum(1 for ch in text if "\u1780" <= ch <= "\u17FF")
    non_space = sum(1 for ch in text if not ch.isspace())
    return khmer_chars / non_space if non_space else 0.0

def langid_audit_pair(pair_key, khmer_threshold=0.5, yo_copy_threshold=0.5):
    src_tr, tgt_tr = DATA[pair_key]["train"]
    src_te, tgt_te = DATA[pair_key]["test"]
    rows = []
    if "km" in pair_key:
        for split, src, tgt in [("train", src_tr, tgt_tr), ("test", src_te, tgt_te)]:
            frac_not_khmer = sum(1 for t in tgt if khmer_script_fraction(t) < khmer_threshold) / len(tgt)
            rows.append({"pair": pair_key, "split": split,
                         "pct_target_not_khmer_script": round(100 * frac_not_khmer, 2)})
    else:
        for split, src, tgt in [("train", src_tr, tgt_tr), ("test", src_te, tgt_te)]:
            n_suspect = sum(1 for s, t in zip(src, tgt)
                             if _jaccard(_char_ngrams(s), _char_ngrams(t)) >= yo_copy_threshold)
            rows.append({"pair": pair_key, "split": split,
                         "pct_target_close_to_source": round(100 * n_suspect / len(tgt), 2)})
    return rows

langid_rows = []
for pk in LANG_PAIRS:
    langid_rows.extend(langid_audit_pair(pk))
langid_df = pd.DataFrame(langid_rows)
print("Target-language-identification audit (script-based proxy, not a full LID model)")
display(langid_df)
langid_df.to_csv(f"{RESULTS_DIR}/table_langid_audit.csv", index=False)


### 3.1 Save raw splits (for transparency / to attach as supplementary data)

In [ ]:
for pk in LANG_PAIRS:
    for split in ["train", "val", "test"]:
        src, tgt = DATA[pk][split]
        with open(f"{DATA_DIR}/{pk}.{split}.src.txt", "w") as f:
            f.write("\n".join(src))
        with open(f"{DATA_DIR}/{pk}.{split}.tgt.txt", "w") as f:
            f.write("\n".join(tgt))
print("Saved raw split files to", DATA_DIR)

## 4. Shared subword tokenisation (vocabulary size set by `BPE_VOCAB`)


In [ ]:
import sentencepiece as spm
import time

# Defined FIRST, before the (potentially slow) tokenizer-training loop below, so these
# always exist even if that loop is interrupted partway through a language pair.
PAD, UNK, BOS, EOS = 0, 1, 2, 3
LSTM_SEPARATE_VOCAB = False
TOKENIZERS = {}

def is_whitespace_segmented(lines, sample=200):
    """Heuristic: does this language actually use spaces between words?
    Khmer/Thai/Chinese/Japanese-style scripts mostly don't, and SentencePiece's
    BPE algorithm searches merges over long unsegmented character runs in that
    case, which is dramatically slower (sometimes prohibitively so) than its
    Unigram algorithm on the same data. Detect this automatically instead of
    hardcoding language codes."""
    sample_lines = lines[:sample]
    if not sample_lines:
        return True
    frac_with_space = sum(1 for l in sample_lines if " " in l) / len(sample_lines)
    return frac_with_space > 0.5

def train_bpe(pair_key):
    src, tgt = DATA[pair_key]["train"]
    corpus_path = f"{DATA_DIR}/{pair_key}.bpe_corpus.txt"
    lines = src + tgt
    with open(corpus_path, "w") as f:
        f.write("\n".join(lines))

    # Cap vocab_size using character volume (works for any script — unlike a
    # whitespace-based word count, which is meaningless for scripts with no spaces)
    # and turn off hard_vocab_limit so training can stop gracefully rather than
    # grinding trying to hit an unreachable exact target.
    n_chars = sum(len(s) for s in lines)
    sane_cap = max(1000, min(BPE_VOCAB, n_chars // 8))
    if sane_cap < BPE_VOCAB:
        print(f"  [{pair_key}] corpus too small for vocab_size={BPE_VOCAB}; capping to {sane_cap}")

   
    model_type = "bpe" if is_whitespace_segmented(tgt) else "unigram"
    if model_type != "bpe":
        print(f"  [{pair_key}] script has no whitespace word boundaries -> using "
              f"model_type='unigram' instead of 'bpe' (much faster/more robust here; "
              f"differs from the BPE used for space-delimited scripts)")

    model_prefix = f"{DATA_DIR}/{pair_key}.bpe"
    t0 = time.time()
    spm.SentencePieceTrainer.train(
        input=corpus_path,
        model_prefix=model_prefix,
        vocab_size=sane_cap,
        model_type=model_type,
        character_coverage=0.9995,
        hard_vocab_limit=False,
        max_sentencepiece_length=8,   # bounds merge-candidate search for no-whitespace scripts
        num_threads=os.cpu_count() or 4,
        pad_id=0, unk_id=1, bos_id=2, eos_id=3,
    )
    print(f"  [{pair_key}] tokenizer trained in {time.time()-t0:.1f}s (model_type={model_type}, vocab_size={sane_cap})")
    sp = spm.SentencePieceProcessor(model_file=f"{model_prefix}.model")
    return sp

for pk in LANG_PAIRS:
    print("Training shared tokenizer for", pk)
    TOKENIZERS[pk] = train_bpe(pk)

def encode(sp, text, max_len=MAX_LEN):
    ids = [BOS] + sp.encode(text, out_type=int)[: max_len - 2] + [EOS]
    return ids

def vocab_coverage(sp, texts):
    unk_count, total = 0, 0
    for t in texts:
        ids = sp.encode(t, out_type=int)
        unk_count += sum(1 for i in ids if i == UNK)
        total += len(ids)
    return 100.0 * (1 - unk_count / max(total, 1))

for pk in LANG_PAIRS:
    src_te, tgt_te = DATA[pk]["test"]
    cov = vocab_coverage(TOKENIZERS[pk], src_te + tgt_te)
    print(f"{pk}: test vocab coverage = {cov:.2f}%")


## 5. PyTorch Dataset / collate

In [8]:
class NMTDataset(torch.utils.data.Dataset):
    def __init__(self, src_texts, tgt_texts, sp, max_len=MAX_LEN):
        self.src = src_texts
        self.tgt = tgt_texts
        self.sp = sp
        self.max_len = max_len

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        s = encode(self.sp, self.src[idx], self.max_len)
        t = encode(self.sp, self.tgt[idx], self.max_len)
        return {"src": s, "tgt": t}

def collate(batch):
    max_s = max(len(b["src"]) for b in batch)
    max_t = max(len(b["tgt"]) for b in batch)
    src = torch.full((len(batch), max_s), PAD, dtype=torch.long)
    tgt = torch.full((len(batch), max_t), PAD, dtype=torch.long)
    for i, b in enumerate(batch):
        src[i, :len(b["src"])] = torch.tensor(b["src"])
        tgt[i, :len(b["tgt"])] = torch.tensor(b["tgt"])
    return src, tgt

## 6. Model 1 — LSTM + Bahdanau Attention

Two-layer bidirectional LSTM encoder, two-layer unidirectional LSTM decoder, additive
(Bahdanau) attention, dropout 0.3.

In [9]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.Wa = nn.Linear(hidden_dim, hidden_dim)
        self.Ua = nn.Linear(hidden_dim, hidden_dim)
        self.va = nn.Linear(hidden_dim, 1)

    def forward(self, dec_hidden, enc_outputs, mask):
        # dec_hidden: [B,H]  enc_outputs: [B,S,H]
        scores = self.va(torch.tanh(self.Wa(dec_hidden).unsqueeze(1) + self.Ua(enc_outputs))).squeeze(-1)
        scores = scores.masked_fill(~mask, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        context = torch.bmm(weights.unsqueeze(1), enc_outputs).squeeze(1)
        return context, weights

class LSTMAttentionNMT(nn.Module):
    def __init__(self, vocab_size, emb_dim=512, hidden_dim=512, dropout=0.3):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.encoder = nn.LSTM(emb_dim, hidden_dim // 2, num_layers=2,
                                bidirectional=True, batch_first=True, dropout=dropout)
        self.decoder = nn.LSTM(emb_dim + hidden_dim, hidden_dim, num_layers=2,
                                batch_first=True, dropout=dropout)
        self.attn = BahdanauAttention(hidden_dim)
        self.out = nn.Linear(hidden_dim * 2, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.hidden_dim = hidden_dim

    def encode(self, src):
        mask = src != PAD
        emb = self.dropout(self.embed(src))
        enc_out, (h, c) = self.encoder(emb)
        # merge bidirectional final states for decoder init
        h = torch.cat([h[-2], h[-1]], dim=-1).unsqueeze(0).repeat(2, 1, 1)
        c = torch.cat([c[-2], c[-1]], dim=-1).unsqueeze(0).repeat(2, 1, 1)
        return enc_out, mask, (h, c)

    def forward(self, src, tgt_input): # Renamed `tgt` to `tgt_input` for clarity
        enc_out, mask, (h, c) = self.encode(src)
        B, T_input = tgt_input.shape # T_input is the length of tgt[:, :-1]
        logits = []
        # The loop should produce T_input predictions, one for each token in tgt_input
        for t in range(T_input):
            dec_input = tgt_input[:, t] # Use current token as input
            emb = self.dropout(self.embed(dec_input)).unsqueeze(1)
            context, _ = self.attn(h[-1], enc_out, mask)
            dec_in_combined = torch.cat([emb, context.unsqueeze(1)], dim=-1)
            out, (h, c) = self.decoder(dec_in_combined, (h, c))
            logit = self.out(torch.cat([out.squeeze(1), context], dim=-1))
            logits.append(logit)
        return torch.stack(logits, dim=1) # [B, T_input, V]

    @torch.no_grad()
    def generate(self, src, max_len=MAX_LEN):
        self.eval()
        enc_out, mask, (h, c) = self.encode(src)
        B = src.size(0)
        dec_input = torch.full((B,), BOS, dtype=torch.long, device=src.device)
        outputs = [dec_input]
        for _ in range(max_len):
            emb = self.embed(dec_input).unsqueeze(1)
            context, _ = self.attn(h[-1], enc_out, mask)
            dec_in = torch.cat([emb, context.unsqueeze(1)], dim=-1)
            out, (h, c) = self.decoder(dec_in, (h, c))
            logit = self.out(torch.cat([out.squeeze(1), context], dim=-1))
            dec_input = logit.argmax(-1)
            outputs.append(dec_input)
        return torch.stack(outputs, dim=1)

## 7. Model 2 — Standard Transformer (Vaswani et al.)

6+6 layers, 8 heads, d_model=512, FFN=2048, tied embeddings, sinusoidal positions, inverse-sqrt
warmup — matching Section V-B. Implemented with `nn.TransformerEncoder/Decoder` for clarity.

In [10]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TransformerNMT(nn.Module):
    def __init__(self, vocab_size, d_model=512, nhead=8, num_layers=6, dim_ff=2048, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD)
        self.pos = PositionalEncoding(d_model)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead, num_encoder_layers=num_layers,
            num_decoder_layers=num_layers, dim_feedforward=dim_ff, dropout=dropout,
            batch_first=True,
        )
        self.out = nn.Linear(d_model, vocab_size)
        self.out.weight = self.embed.weight  # tied embeddings
        self.d_model = d_model

    def make_masks(self, src, tgt):
        src_pad = src == PAD
        tgt_pad = tgt == PAD
        causal = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(src.device).bool() # Explicitly convert to bool
        return src_pad, tgt_pad, causal

    def forward(self, src, tgt_in):
        src_pad, tgt_pad, causal = self.make_masks(src, tgt_in)
        s = self.pos(self.embed(src) * math.sqrt(self.d_model))
        t = self.pos(self.embed(tgt_in) * math.sqrt(self.d_model))
        out = self.transformer(s, t, src_key_padding_mask=src_pad, tgt_key_padding_mask=tgt_pad,
                                tgt_mask=causal, memory_key_padding_mask=src_pad)
        return self.out(out)

    @torch.no_grad()
    def generate(self, src, max_len=MAX_LEN):
        self.eval()
        B = src.size(0)
        ys = torch.full((B, 1), BOS, dtype=torch.long, device=src.device)
        for _ in range(max_len):
            logits = self.forward(src, ys)
            next_tok = logits[:, -1].argmax(-1, keepdim=True)
            ys = torch.cat([ys, next_tok], dim=1)
        return ys

## 8. Model 3 — Transformer with multilingual pretraining (mBERT encoder)


In [ ]:
def build_pretrained_encdec():
    model = EncoderDecoderModel.from_encoder_decoder_pretrained(
        "bert-base-multilingual-cased", "bert-base-multilingual-cased"
    )
    # re-init decoder weights so that only the ENCODER carries pretrained weights
    def reinit(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            m.weight.data.normal_(mean=0.0, std=0.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                m.bias.data.zero_()
    model.decoder.apply(reinit)

    # Explicitly set generation configuration tokens
    model.config.decoder_start_token_id = mbert_tokenizer.cls_token_id # [CLS] token for BERT (101)
    model.config.pad_token_id = mbert_tokenizer.pad_token_id # [PAD] token for BERT (0)
    model.config.bos_token_id = mbert_tokenizer.cls_token_id # Use [CLS] as beginning of sequence
    model.config.eos_token_id = mbert_tokenizer.sep_token_id # Use [SEP] as end of sequence
    model.config.vocab_size = model.config.decoder.vocab_size

    # IMPORTANT: recent `transformers` versions read generation settings from the separate
    # `model.generation_config` object, not `model.config`, at .generate() time. That object
    # is populated from model.config when the model is first loaded -- setting model.config
    # afterward (as above) does NOT retroactively update it, which is exactly what caused
    # "decoder_start_token_id or bos_token_id has to be defined for encoder-decoder generation".
    # Mirror the same values onto generation_config explicitly so .generate() actually sees them.
    model.generation_config.decoder_start_token_id = mbert_tokenizer.cls_token_id
    model.generation_config.pad_token_id = mbert_tokenizer.pad_token_id
    model.generation_config.bos_token_id = mbert_tokenizer.cls_token_id
    model.generation_config.eos_token_id = mbert_tokenizer.sep_token_id
    return model

mbert_tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

## 9. Model 4 — mT5-style text-to-text model

Fine-tunes `google/mt5-small` (~300M params, 12+12 layers, d_model=768, 250k SentencePiece
vocab, pretrained on 101 languages), a small multilingual text-to-text model.
Framed as text-to-text with an explicit direction prefix.

In [12]:
MT5_CHECKPOINT = "google/mt5-small"
mt5_tokenizer = AutoTokenizer.from_pretrained(MT5_CHECKPOINT)

def build_mt5():
    return MT5ForConditionalGeneration.from_pretrained(MT5_CHECKPOINT)

DIRECTION_PREFIX = {
    "en-yo": "translate English to Yoruba: ",
    "en-km": "translate English to Khmer: ",
}

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

## 10. Generic training loop (custom PyTorch models) with early stopping

Mirrors Section V-E: evaluate every `EVAL_EVERY` steps, patience of `PATIENCE` checkpoints,
Adam optimiser, label smoothing 0.1, inverse-sqrt warmup for the Transformer family.

In [ ]:
def compute_bleu(hyps, refs):
    return sacrebleu.corpus_bleu(hyps, [refs]).score

def decode_ids(sp_or_hf, ids, is_hf=False):
    if is_hf:
        return sp_or_hf.decode(ids, skip_special_tokens=True)
    ids = [i for i in ids if i not in (PAD, BOS, EOS)]
    return sp_or_hf.decode(ids)

def train_custom_model(model_type, pair_key, seed, use_bpe=True, use_pretrain=False):
    set_seed(seed)
    sp = TOKENIZERS[pair_key]
    vocab_size = sp.get_piece_size()
    src_tr, tgt_tr = DATA[pair_key]["train"]
    src_va, tgt_va = DATA[pair_key]["val"]
    src_te, tgt_te = DATA[pair_key]["test"]

    train_ds = NMTDataset(src_tr, tgt_tr, sp)
    val_ds = NMTDataset(src_va, tgt_va, sp)
    test_ds = NMTDataset(src_te, tgt_te, sp)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
    val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, collate_fn=collate)

    if model_type == "LSTM-Attention":
        model = LSTMAttentionNMT(vocab_size).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
        scheduler = None
    else:
        model = TransformerNMT(vocab_size).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.98), eps=1e-9)
        scheduler = get_inverse_sqrt_schedule(optimizer, num_warmup_steps=4000)

    criterion = nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=0.1)

    step, best_bleu, patience_ct = 0, -1, 0
    history = {"step": [], "train_loss": [], "val_loss": []}
    t0 = time.time()
    model.train()
    stop = False
    while step < MAX_STEPS and not stop:
        for src, tgt in train_dl:
            src, tgt = src.to(device), tgt.to(device)
            optimizer.zero_grad()
            logits = model(src, tgt[:, :-1])
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            if scheduler: scheduler.step()
            step += 1

            if step % EVAL_EVERY == 0 or step == MAX_STEPS:
                model.eval()
                val_loss_total, val_n = 0.0, 0
                with torch.no_grad():
                    for vs, vt in val_dl:
                        vs, vt = vs.to(device), vt.to(device)
                        vlogits = model(vs, vt[:, :-1])
                        vloss = criterion(vlogits.reshape(-1, vlogits.size(-1)), vt[:, 1:].reshape(-1))
                        val_loss_total += vloss.item() * vs.size(0); val_n += vs.size(0)
                val_loss = val_loss_total / max(val_n, 1)
                history["step"].append(step)
                history["train_loss"].append(loss.item())
                history["val_loss"].append(val_loss)

                hyps = translate_custom(model, sp, src_va[:200])
                bleu = compute_bleu(hyps, tgt_va[:200])
                print(f"  [{model_type}][{pair_key}][seed={seed}] Step {step}/{MAX_STEPS} | Train Loss: {loss.item():.2f} | Val Loss: {val_loss:.2f} | Val BLEU: {bleu:.2f} | Patience: {patience_ct}/{PATIENCE}")

                if bleu > best_bleu:
                    best_bleu = bleu; patience_ct = 0
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                else:
                    patience_ct += 1
                model.train()
                if patience_ct >= PATIENCE or step >= MAX_STEPS:
                    stop = True
                    break
    model.load_state_dict(best_state)
    gpu_hours = (time.time() - t0) / 3600.0

    hyps_test = translate_custom(model, sp, src_te)
    test_bleu = compute_bleu(hyps_test, tgt_te)
    n_params = sum(p.numel() for p in model.parameters())
    del model; gc.collect(); torch.cuda.empty_cache()
    return {"test_bleu": test_bleu, "history": history, "gpu_hours": gpu_hours,
            "n_params_m": n_params / 1e6, "hyps": hyps_test}

@torch.no_grad()
def translate_custom(model, sp, texts, batch_size=32):
    model.eval()
    hyps = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        max_s = max(len(encode(sp, t)) for t in batch)
        src = torch.full((len(batch), max_s), PAD, dtype=torch.long)
        for j, t in enumerate(batch):
            ids = encode(sp, t)
            src[j, :len(ids)] = torch.tensor(ids)
        src = src.to(device)
        out_ids = model.generate(src)
        for row in out_ids.tolist():
            hyps.append(decode_ids(sp, row))
    return hyps

## 11. Training routine for HF pretrained models (mBERT enc-dec, mT5)

In [14]:
def make_hf_datasets(pair_key, tokenizer, prefix="", max_len=MAX_LEN):
    src_tr, tgt_tr = DATA[pair_key]["train"]
    src_va, tgt_va = DATA[pair_key]["val"]
    src_te, tgt_te = DATA[pair_key]["test"]

    def tok(src_list, tgt_list):
        model_inputs = tokenizer([prefix + s for s in src_list], max_length=max_len,
                                  truncation=True, padding=False)
        with tokenizer.as_target_tokenizer() if hasattr(tokenizer, "as_target_tokenizer") else _nullctx():
            labels = tokenizer(tgt_list, max_length=max_len, truncation=True, padding=False)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    from datasets import Dataset as HFDataset
    train_ds = HFDataset.from_dict({"src": src_tr, "tgt": tgt_tr}).map(
        lambda ex: tok(ex["src"], ex["tgt"]), batched=True, remove_columns=["src", "tgt"])
    val_ds = HFDataset.from_dict({"src": src_va, "tgt": tgt_va}).map(
        lambda ex: tok(ex["src"], ex["tgt"]), batched=True, remove_columns=["src", "tgt"])
    test_texts = (src_te, tgt_te)
    return train_ds, val_ds, test_texts

import contextlib
@contextlib.contextmanager
def _nullctx():
    yield

def train_hf_seq2seq(model, tokenizer, pair_key, seed, prefix="", max_steps=MAX_STEPS, model_tag="model"):
    set_seed(seed)
    train_ds, val_ds, (src_te, tgt_te) = make_hf_datasets(pair_key, tokenizer, prefix)
    collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

    args = Seq2SeqTrainingArguments(
        output_dir=f"./ckpt_{model_tag}_{pair_key}_{seed}",
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        max_steps=max_steps,
        eval_strategy="steps",
        eval_steps=EVAL_EVERY,
        save_strategy="steps",
        save_steps=EVAL_EVERY,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        # mT5/T5-family models are known to collapse into a degenerate, low-entropy
        # output (e.g. repeating one token) when fine-tuned with plain AdamW at high
        # learning rates without warmup. A lower learning rate (1e-4) and an explicit
        # warmup are used here to avoid that failure mode.
        learning_rate=1e-4 if "mt5" in tokenizer.name_or_path.lower() else 5e-5,
        warmup_steps=int(0.1 * max_steps) if "mt5" in tokenizer.name_or_path.lower() else 0,
        predict_with_generate=True,
        generation_max_length=MAX_LEN,
        logging_steps=EVAL_EVERY,
        fp16=torch.cuda.is_available(),
        report_to=[],
    )
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                              data_collator=collator)
    t0 = time.time()
    # If this exact run crashed/restarted partway through, resume from its last saved HF
    # Trainer checkpoint (saved every EVAL_EVERY steps in output_dir) instead of starting
    # this run over from step 0 -- important on a run that may take many hours.
    import glob
    existing_ckpts = glob.glob(f"{args.output_dir}/checkpoint-*")
    resume = bool(existing_ckpts)
    if resume:
        print(f"  Found existing checkpoint(s) in {args.output_dir}, resuming from the latest.")
    trainer.train(resume_from_checkpoint=resume)
    gpu_hours = (time.time() - t0) / 3600.0

    hyps = []
    model.eval()
    for i in range(0, len(src_te), BATCH_SIZE):
        batch = [prefix + s for s in src_te[i:i + BATCH_SIZE]]
        enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to(device)
        with torch.no_grad():
            gen = model.generate(**enc, max_length=MAX_LEN)
        hyps.extend(tokenizer.batch_decode(gen, skip_special_tokens=True))
    test_bleu = compute_bleu(hyps, tgt_te)
    n_params = sum(p.numel() for p in model.parameters())

    log_history = trainer.state.log_history
    train_loss = [h["loss"] for h in log_history if "loss" in h]
    val_loss = [h["eval_loss"] for h in log_history if "eval_loss" in h]
    history = {"train_loss": train_loss, "val_loss": val_loss}

    del trainer, model; gc.collect(); torch.cuda.empty_cache()
    return {"test_bleu": test_bleu, "history": history, "gpu_hours": gpu_hours,
            "n_params_m": n_params / 1e6, "hyps": hyps}

 12. Run the full experiment grid


In [ ]:
import os, json as _json

CKPT_PATH = f"{RESULTS_DIR}/raw_results_checkpoint.jsonl"

def _load_checkpoint():
    done = {}
    if os.path.exists(CKPT_PATH):
        with open(CKPT_PATH) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                r = _json.loads(line)
                key = (r["model"], r["pair"], r["seed"])
                done[key] = r
    return done

def _append_checkpoint(r):
    # Written and flushed immediately after each run completes, so a Colab disconnect
    # or VM recycle only costs the one run in progress, not the whole grid. Re-running
    # this cell after a restart skips everything already in the checkpoint file.
    # NOTE: unlike raw_results.json below, this stores the FULL record (including hyps
    # and history) -- downstream analysis cells (loss-curve plots, OOV-rate table,
    # sample-hypothesis export) need those fields even for runs loaded from a resume.
    with open(CKPT_PATH, "a") as f:
        f.write(_json.dumps(r) + "\n")
        f.flush()
        os.fsync(f.fileno())

_done = _load_checkpoint()
if _done:
    print(f"Resuming: {len(_done)} run(s) already completed in {CKPT_PATH}, will be skipped.")

results = list(_done.values())  # list of dicts: model, pair, seed, test_bleu, gpu_hours, n_params_m, hyps, history

for pair_key in LANG_PAIRS:
    for seed in SEEDS:
        # 1. LSTM-Attention
        key = ("LSTM-Attention", pair_key, seed)
        if "LSTM-Attention" not in MODEL_NAMES_TO_RUN:
            pass
        elif key in _done:
            print(f"[skip, already done] LSTM-Attention / {pair_key} / seed={seed}")
        else:
            print(f"\n--- Starting LSTM-Attention for {pair_key} (seed={seed}) ---")
            r = train_custom_model("LSTM-Attention", pair_key, seed)
            r = {"model": "LSTM-Attention", "pair": pair_key, "seed": seed, **r}
            results.append(r); _append_checkpoint(r); _done[key] = r
            print(f"[LSTM-Attention][{pair_key}][seed={seed}] BLEU={r['test_bleu']:.2f}")

        # 2. Standard Transformer
        key = ("Transformer", pair_key, seed)
        if "Transformer" not in MODEL_NAMES_TO_RUN:
            pass
        elif key in _done:
            print(f"[skip, already done] Transformer / {pair_key} / seed={seed}")
        else:
            print(f"\n--- Starting Standard Transformer for {pair_key} (seed={seed}) ---")
            r = train_custom_model("Transformer", pair_key, seed)
            r = {"model": "Transformer", "pair": pair_key, "seed": seed, **r}
            results.append(r); _append_checkpoint(r); _done[key] = r
            print(f"[Transformer][{pair_key}][seed={seed}] BLEU={r['test_bleu']:.2f}")

        # 3. Transformer + mBERT pretraining
        key = ("Transformer+Pretraining", pair_key, seed)
        if "Transformer+Pretraining" not in MODEL_NAMES_TO_RUN:
            pass
        elif key in _done:
            print(f"[skip, already done] Transformer+Pretraining / {pair_key} / seed={seed}")
        else:
            print(f"\n--- Starting Transformer + mBERT pretraining for {pair_key} (seed={seed}) ---")
            model = build_pretrained_encdec().to(device)
            r = train_hf_seq2seq(model, mbert_tokenizer, pair_key, seed, model_tag="mbert")
            r = {"model": "Transformer+Pretraining", "pair": pair_key, "seed": seed, **r}
            results.append(r); _append_checkpoint(r); _done[key] = r
            print(f"[Transformer+Pretraining][{pair_key}][seed={seed}] BLEU={r['test_bleu']:.2f}")

        # 4. mT5-style
        key = ("mT5-Style", pair_key, seed)
        if "mT5-Style" not in MODEL_NAMES_TO_RUN:
            pass
        elif key in _done:
            print(f"[skip, already done] mT5-Style / {pair_key} / seed={seed}")
        else:
            print(f"\n--- Starting mT5-Style for {pair_key} (seed={seed}) ---")
            model = build_mt5().to(device)
            prefix = DIRECTION_PREFIX[pair_key]
            r = train_hf_seq2seq(model, mt5_tokenizer, pair_key, seed, prefix=prefix, model_tag="mt5")
            r = {"model": "mT5-Style", "pair": pair_key, "seed": seed, **r}
            results.append(r); _append_checkpoint(r); _done[key] = r
            print(f"[mT5-Style][{pair_key}][seed={seed}] BLEU={r['test_bleu']:.2f}")

with open(f"{RESULTS_DIR}/raw_results.json", "w") as f:
    json.dump([{k: v for k, v in r.items() if k not in ("hyps", "history")}
                for r in results], f, indent=2)
print("Saved raw_results.json")


## Before/after-filtering comparison (En-Yo) — re-trains on cleaned data to directly test whether the high BLEU is contamination-driven

In [ ]:

FILTER_PAIR = "en-yo"
NEAR_DUP_THRESHOLD = 0.7
COPY_THRESHOLD = 0.5

def build_filtered_data(pair_key):
    src_tr, tgt_tr = DATA[pair_key]["train"]
    src_te, tgt_te = DATA[pair_key]["test"]
    train_ngrams = [_char_ngrams(s) for s in src_tr]

    # filter TEST: drop examples whose source is a near-duplicate of any train source,
    # or whose target is very close to its own source (likely untranslated copy)
    keep_te = []
    for i, (s, t) in enumerate(zip(src_te, tgt_te)):
        te_ng = _char_ngrams(s)
        is_near_dup = any(_jaccard(te_ng, tn) >= NEAR_DUP_THRESHOLD for tn in train_ngrams)
        is_copy = _jaccard(_char_ngrams(s), _char_ngrams(t)) >= COPY_THRESHOLD
        if not is_near_dup and not is_copy:
            keep_te.append(i)

    # filter TRAIN: drop examples whose target is very close to its own source
    keep_tr = [i for i, (s, t) in enumerate(zip(src_tr, tgt_tr))
               if _jaccard(_char_ngrams(s), _char_ngrams(t)) < COPY_THRESHOLD]

    filt_src_tr = [src_tr[i] for i in keep_tr]; filt_tgt_tr = [tgt_tr[i] for i in keep_tr]
    filt_src_te = [src_te[i] for i in keep_te]; filt_tgt_te = [tgt_te[i] for i in keep_te]
    print(f"{pair_key}: train {len(src_tr)} -> {len(filt_src_tr)} "
          f"({len(src_tr)-len(filt_src_tr)} removed), "
          f"test {len(src_te)} -> {len(filt_src_te)} ({len(src_te)-len(filt_src_te)} removed)")
    return (filt_src_tr, filt_tgt_tr), (filt_src_te, filt_tgt_te)

(filt_train, filt_test) = build_filtered_data(FILTER_PAIR)

filter_results = []
_orig_train = DATA[FILTER_PAIR]["train"]
_orig_test = DATA[FILTER_PAIR]["test"]
try:
    DATA[FILTER_PAIR]["train"] = filt_train
    DATA[FILTER_PAIR]["test"] = filt_test
    for model_type in ["LSTM-Attention", "Transformer"]:
        before_row = next((r for r in results if r["model"] == model_type
                            and r["pair"] == FILTER_PAIR and r["seed"] == SEEDS[0]), None)
        print(f"\n--- Re-training {model_type} on filtered {FILTER_PAIR} ---")
        r_after = train_custom_model(model_type, FILTER_PAIR, SEEDS[0])
        filter_results.append({
            "model": model_type,
            "bleu_before_filtering": before_row["test_bleu"] if before_row else None,
            "bleu_after_filtering": r_after["test_bleu"],
        })
finally:
    DATA[FILTER_PAIR]["train"] = _orig_train
    DATA[FILTER_PAIR]["test"] = _orig_test

filter_df = pd.DataFrame(filter_results)
print(f"\n{FILTER_PAIR} BLEU before vs. after contamination filtering")
display(filter_df)
filter_df.to_csv(f"{RESULTS_DIR}/table_filtering_before_after.csv", index=False)


## Debug: inspect raw mT5-Style hypotheses (confirms whether output is empty/degenerate)

In [ ]:

mt5_rows = [r for r in results if r["model"] == "mT5-Style"]
if not mt5_rows:
    print("No mT5-Style results found in `results` -- reload from the checkpoint first.")
else:
    r = mt5_rows[0]
    print(f"Inspecting mT5-Style / {r['pair']} / seed={r['seed']}")
    print(f"Reported test_bleu: {r['test_bleu']:.4f}\n")
    hyps = r["hyps"]
    refs = DATA[r["pair"]]["test"][1]
    print(f"{len(hyps)} hypotheses total. First 10 (raw, unfiltered):\n")
    for i in range(min(10, len(hyps))):
        h = hyps[i]
        print(f"  [{i}] hyp={h!r}")
        print(f"       ref={refs[i]!r}")
    empty = sum(1 for h in hyps if len(h.strip()) == 0)
    unique = len(set(hyps))
    print(f"\nEmpty hypotheses: {empty}/{len(hyps)}")
    print(f"Unique hypotheses: {unique}/{len(hyps)} (a very low number here means degenerate/repetitive output)")


## 13. Main BLEU results (mean over seeds)

In [ ]:
df = pd.DataFrame(results)
table3 = df.pivot_table(index="model", columns="pair", values="test_bleu", aggfunc="mean")
table3["Avg"] = table3.mean(axis=1)
gpu = df.groupby("model")["gpu_hours"].mean()
table3["GPU hrs"] = gpu
table3 = table3.reindex(MODEL_NAMES)
print("BLEU Results Across Low-Resource Language Pairs")
display(table3.round(2))
table3.to_csv(f"{RESULTS_DIR}/table3_bleu_results.csv")

## 13B. chrF++ results (computed from saved hypotheses, no retraining needed)

In [ ]:
def compute_chrf(hyps, refs):
    # chrF++ = chrF with word_order=2 (adds word unigram/bigram F-score alongside the
    # default character n-grams) -- the standard "chrF++" variant used in MT literature.
    return sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score

# Reuses hyps already stored in `results` (from a fresh run, or reloaded from
# results/raw_results_checkpoint.jsonl) -- no retraining required.
for r in results:
    if "test_chrf" not in r:
        refs = DATA[r["pair"]]["test"][1]
        r["test_chrf"] = compute_chrf(r["hyps"], refs)

df_chrf = pd.DataFrame(results)
table3b = df_chrf.pivot_table(index="model", columns="pair", values="test_chrf", aggfunc="mean")
table3b["Avg"] = table3b.mean(axis=1)
table3b = table3b.reindex(MODEL_NAMES)
print("chrF++ Results Across Low-Resource Language Pairs")
display(table3b.round(2))
table3b.to_csv(f"{RESULTS_DIR}/table3b_chrf_results.csv")


## 14. Statistical significance vs. LSTM-Attention baseline (paired t-test)

In [ ]:
sig_rows = []
baseline = df[df.model == "LSTM-Attention"].groupby("pair")["test_bleu"].apply(list)
for model_name in MODEL_NAMES[1:]:
    comp = df[df.model == model_name].groupby("pair")["test_bleu"].apply(list)
    all_base, all_comp = [], []
    for pk in LANG_PAIRS:
        all_base.extend(baseline.get(pk, []))
        all_comp.extend(comp.get(pk, []))
    if len(all_base) >= 2 and len(all_comp) >= 2 and len(all_base) == len(all_comp):
        t_stat, p_val = stats.ttest_rel(all_comp, all_base)
    else:
        t_stat, p_val = float("nan"), float("nan")
    sig_rows.append({"Comparison": f"{model_name} vs. LSTM-Attention", "t-statistic": t_stat, "p-value": p_val})

table6 = pd.DataFrame(sig_rows)
print("Statistical Significance (paired t-test across seeds/pairs)")
display(table6)
table6.to_csv(f"{RESULTS_DIR}/table6_significance.csv", index=False)
print("\nNote: with only", len(SEEDS), "seed(s) per condition in REDUCED mode, this test is under-powered.")
print("Run with 5 seeds (FULL_SCALE) before interpreting these p-values.")

## Statistical rigor: paired bootstrap significance and per-seed std

In [ ]:

import numpy as np

def paired_bootstrap_bleu(hyps_a, hyps_b, refs, n_bootstrap=1000, seed=13):
    rng = np.random.RandomState(seed)
    n = len(refs)
    wins_b, diffs = 0, []
    for _ in range(n_bootstrap):
        idx = rng.randint(0, n, n)
        h_a = [hyps_a[i] for i in idx]; h_b = [hyps_b[i] for i in idx]; r = [refs[i] for i in idx]
        bleu_a = sacrebleu.corpus_bleu(h_a, [r]).score
        bleu_b = sacrebleu.corpus_bleu(h_b, [r]).score
        diffs.append(bleu_b - bleu_a)
        if bleu_b > bleu_a:
            wins_b += 1
    diffs = np.array(diffs)
    p_value = 2 * min(wins_b, n_bootstrap - wins_b) / n_bootstrap  # two-sided
    return {
        "mean_diff": diffs.mean(),
        "ci95_low": np.percentile(diffs, 2.5),
        "ci95_high": np.percentile(diffs, 97.5),
        "p_value": p_value,
    }

# One (model, pair) comparison vs. the LSTM-Attention baseline, per available seed.
bootstrap_rows = []
for pk in LANG_PAIRS:
    base_runs = {r["seed"]: r for r in results if r["model"] == "LSTM-Attention" and r["pair"] == pk}
    for model_name in MODEL_NAMES:
        if model_name == "LSTM-Attention":
            continue
        comp_runs = {r["seed"]: r for r in results if r["model"] == model_name and r["pair"] == pk}
        for sd in sorted(set(base_runs) & set(comp_runs)):
            base_r, comp_r = base_runs[sd], comp_runs[sd]
            refs = DATA[pk]["test"][1]
            stats = paired_bootstrap_bleu(base_r["hyps"], comp_r["hyps"], refs, seed=sd)
            bootstrap_rows.append({"model": model_name, "pair": pk, "seed": sd, **stats})

bootstrap_df = pd.DataFrame(bootstrap_rows)
print("Paired bootstrap resampling significance (Koehn, 2004), vs. LSTM-Attention baseline")
display(bootstrap_df.round(4))
bootstrap_df.to_csv(f"{RESULTS_DIR}/table_bootstrap_significance.csv", index=False)

# Per-seed mean/std BLEU, for full transparency alongside the point estimates in the main BLEU table.
# NOTE: with only 2 seeds this std is itself a rough, low-precision estimate; interpret with
# caution and use more seeds where possible.
seed_stats = (pd.DataFrame(results).groupby(["model", "pair"])["test_bleu"]
              .agg(["mean", "std", "count"]).reset_index())
print("\nPer-(model,pair) BLEU mean/std across available seeds")
display(seed_stats.round(2))
seed_stats.to_csv(f"{RESULTS_DIR}/table_seed_mean_std.csv", index=False)


## 15. Ablation study — BPE vs. no-BPE, pretrained vs. non-pretrained



In [ ]:
def train_transformer_word_level(pair_key, seed, use_bpe):
    # quick ablation helper: reuses train_custom_model but swaps tokenizer granularity
    if use_bpe:
        return train_custom_model("Transformer", pair_key, seed)
    # crude whitespace/word-level fallback tokenizer for the ablation-only comparison
    src_tr, tgt_tr = DATA[pair_key]["train"]
    vocab = sorted(set(w for s in (src_tr + tgt_tr) for w in s.split()))[:8000]
    class WordSP:
        def __init__(self, vocab):
            self.stoi = {w: i + 4 for i, w in enumerate(vocab)}
            self.itos = {i: w for w, i in self.stoi.items()}
        def encode(self, text, out_type=int):
            return [self.stoi.get(w, UNK) for w in text.split()]
        def decode(self, ids):
            return " ".join(self.itos.get(i, "") for i in ids if i in self.itos)
        def get_piece_size(self):
            return len(self.stoi) + 4
    TOKENIZERS[pair_key + "_word"] = WordSP(vocab)
    saved = TOKENIZERS[pair_key]
    TOKENIZERS[pair_key] = TOKENIZERS[pair_key + "_word"]
    r = train_custom_model("Transformer", pair_key, seed)
    TOKENIZERS[pair_key] = saved
    return r

ablation_pair, ablation_seed = next(iter(LANG_PAIRS)), SEEDS[0]  # first configured pair
r_no_bpe = train_transformer_word_level(ablation_pair, ablation_seed, use_bpe=False)
r_bpe = train_transformer_word_level(ablation_pair, ablation_seed, use_bpe=True)
# NOTE: the pretrained-model rows (mBERT encoder-decoder, mT5) from the original design
# are intentionally omitted here -- Transformer+Pretraining and mT5-Style were dropped
# from this run entirely (infeasible on available CPU hardware: 46h+ / 300h+ ETAs
# observed), so this ablation only covers tokenization granularity, not pretraining.

table4 = pd.DataFrame([
    {"Configuration": "Transformer (word-level tokenization)", "BLEU": r_no_bpe["test_bleu"]},
    {"Configuration": "Transformer (BPE tokenization)", "BLEU": r_bpe["test_bleu"]},
])
print(f"Tokenization ablation on {ablation_pair} (single seed slice)")
print("(Pretraining-related rows omitted: Transformer+Pretraining/mT5-Style were not run.)")
display(table4)
table4.to_csv(f"{RESULTS_DIR}/table4_ablation.csv", index=False)


In [ ]:
plt.figure(figsize=(7, 5))
sns.barplot(data=table4, x="Configuration", y="BLEU", hue="Configuration", legend=False, palette="viridis")
plt.xticks(rotation=20, ha="right")
plt.ylabel("BLEU")
plt.title("Ablation: tokenization granularity")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig4_ablation.png", dpi=150)
plt.show()


## 16. BLEU bar chart and heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_df = table3.drop(columns=["Avg", "GPU hrs"]).reset_index()
plot_df_melt = plot_df.melt(id_vars="model", var_name="pair", value_name="BLEU")
sns.barplot(data=plot_df_melt, x="pair", y="BLEU", hue="model", ax=axes[0],
            hue_order=MODEL_NAMES)
axes[0].set_title("BLEU by model and language pair")
axes[0].legend(fontsize=8)

heat_data = table3.drop(columns=["Avg", "GPU hrs"]).reindex(MODEL_NAMES)
sns.heatmap(heat_data, annot=True, fmt=".1f", cmap="viridis", ax=axes[1])
axes[1].set_title("BLEU heatmap")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig1_fig2_bleu.png", dpi=150)
plt.show()

## 17. Training/validation loss curves

In [ ]:
plt.figure(figsize=(9, 6))
_first_pair = next(iter(LANG_PAIRS))
for r in results:
    if r["pair"] != _first_pair or r["seed"] != SEEDS[0]:
        continue
    h = r["history"]
    if not h.get("train_loss"):
        continue
    # Custom models (LSTM/Transformer) log loss per training step; HF Trainer's
    # log_history (used for mBERT/mT5) doesn't record step numbers the same way,
    # so fall back to a plain per-log-event index for those.
    xs = h["step"] if h.get("step") else list(range(1, len(h["train_loss"]) + 1))
    plt.plot(xs, h["train_loss"], label=f"{r['model']} train (seed {r['seed']})")
    if h.get("val_loss"):
        xs_val = h["step"] if h.get("step") else list(range(1, len(h["val_loss"]) + 1))
        plt.plot(xs_val, h["val_loss"], "--", label=f"{r['model']} val (seed {r['seed']})")
plt.xlabel("Training step / logging event"); plt.ylabel("Loss")
plt.title(f"Training dynamics, all models ({_first_pair})")
plt.legend(fontsize=7)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig3_training_curves.png", dpi=150)
plt.show()


## 18. Quality vs. compute efficiency frontier

In [ ]:
eff = table3.reset_index()
plt.figure(figsize=(7, 5))
plt.scatter(eff["GPU hrs"], eff["Avg"], s=80)
for _, row in eff.iterrows():
    plt.annotate(row["model"], (row["GPU hrs"], row["Avg"]), textcoords="offset points", xytext=(5, 5))
order = eff.sort_values("GPU hrs")
plt.plot(order["GPU hrs"], order["Avg"], "--", alpha=0.5)
plt.xlabel("GPU hours"); plt.ylabel("Average BLEU")
plt.title("Quality vs. compute efficiency frontier")
plt.savefig(f"{RESULTS_DIR}/fig5_efficiency.png", dpi=150)
plt.show()

## 19. Model scale vs. BLEU (regression)

In [ ]:
params_df = df.groupby("model")["n_params_m"].mean().reindex(MODEL_NAMES)
scale_df = pd.DataFrame({"params_m": params_df, "avg_bleu": table3["Avg"]}).reset_index()

slope, intercept, r_value, p_value, std_err = stats.linregress(scale_df["params_m"], scale_df["avg_bleu"])
plt.figure(figsize=(7, 5))
plt.scatter(scale_df["params_m"], scale_df["avg_bleu"], s=80)
for _, row in scale_df.iterrows():
    plt.annotate(row["model"], (row["params_m"], row["avg_bleu"]), textcoords="offset points", xytext=(5, 5))
xs = np.linspace(scale_df["params_m"].min(), scale_df["params_m"].max(), 50)
plt.plot(xs, slope * xs + intercept, "--", color="gray", label=f"r={r_value:.2f}")
plt.xlabel("Parameters (M)"); plt.ylabel("Average BLEU")
plt.title("Model scale vs. translation quality")
plt.legend()
plt.savefig(f"{RESULTS_DIR}/fig6_scale_vs_bleu.png", dpi=150)
plt.show()
print(f"Pearson r = {r_value:.3f}, p = {p_value:.4f}")

## 20. BLEU distribution across seeds (box plots)



In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="model", y="test_bleu", order=MODEL_NAMES)
sns.stripplot(data=df, x="model", y="test_bleu", order=MODEL_NAMES, color="black", alpha=0.5, jitter=True)
plt.title("BLEU distribution across seeds and language pairs")
plt.xticks(rotation=15)
plt.savefig(f"{RESULTS_DIR}/fig10_bleu_distribution.png", dpi=150)
plt.show()

## 21. Automated proxy for qualitative error analysis



In [ ]:
def oov_rate(hyp, sp):
    ids = sp.encode(hyp, out_type=int)
    return 100.0 * sum(1 for i in ids if i == UNK) / max(len(ids), 1)

def length_ratio_disfluency(hyp, ref):
    lh, lr = len(hyp.split()), max(len(ref.split()), 1)
    return abs(lh - lr) / lr

proxy_rows = []
for r in results:
    pk = r["pair"]; sp = TOKENIZERS[pk]
    _, tgt_te = DATA[pk]["test"]
    hyps = r["hyps"]
    oov = np.mean([oov_rate(h, sp) for h in hyps])
    disfluency = np.mean([length_ratio_disfluency(h, t) for h, t in zip(hyps, tgt_te)])
    proxy_rows.append({"model": r["model"], "pair": pk, "seed": r["seed"],
                        "oov_proxy_pct": oov, "length_disfluency_proxy": disfluency})

proxy_df = pd.DataFrame(proxy_rows)
proxy_summary = proxy_df.groupby("model")[["oov_proxy_pct", "length_disfluency_proxy"]].mean().reindex(MODEL_NAMES)
print("Automated error-analysis PROXY (not a substitute for human annotation):")
display(proxy_summary.round(2))
proxy_summary.to_csv(f"{RESULTS_DIR}/table5_error_proxy.csv")

## Data-scale curve — effect of training-set size

Trains at N=1k/4k/16k/full on one pair to measure the marginal value of more training data, to examine the fixed-budget design choice.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for model_type in DATA_SCALE_MODELS:
    sub = scale_df[scale_df["model"] == model_type].sort_values("N_TRAIN")
    ax.plot(sub["N_TRAIN"], sub["test_bleu"], marker="o", label=model_type)
ax.set_xscale("log")
ax.set_xlabel("Training-set size (N_TRAIN, log scale)")
ax.set_ylabel("Test BLEU")
ax.set_title(f"Data-scale curve ({DATA_SCALE_PAIR})")
ax.legend()
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig12_data_scale_curve.png", dpi=150)
plt.show()


## Dataset audit: contamination, train–test near-duplicate overlap, short segments

Directly checks whether train/test overlap (rather than genuine translation quality) could explain unusually high BLEU on any pair -- e.g. the En-Yo result.

In [ ]:

def _short_seg_pct(texts, max_tokens=5):
    if not texts:
        return 0.0
    return 100.0 * sum(1 for t in texts if len(t.split()) <= max_tokens) / len(texts)

def _exact_dup_rate(train_src, eval_src):
    if not eval_src:
        return 0.0
    train_set = set(train_src)
    return 100.0 * sum(1 for s in eval_src if s in train_set) / len(eval_src)

def _jaccard(a, b):
    sa, sb = set(a.split()), set(b.split())
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)

def _near_dup_rate(train_src, eval_src, threshold=0.8, cap_train_check=3000):
    if not eval_src:
        return 0.0
    # cap the train side searched per example to keep this tractable on large pairs;
    # a random (seeded) sample is representative for an audit statistic, not exhaustive
    rng = random.Random(42)
    train_sample = train_src if len(train_src) <= cap_train_check else rng.sample(train_src, cap_train_check)
    hits = 0
    for s in eval_src:
        if any(_jaccard(s, t) >= threshold for t in train_sample):
            hits += 1
    return 100.0 * hits / len(eval_src)

audit_rows = []
for pk in LANG_PAIRS:
    train_src, _ = DATA[pk]["train"]
    val_src, _ = DATA[pk]["val"]
    test_src, _ = DATA[pk]["test"]
    all_src = train_src + val_src + test_src

    audit_rows.append({
        "pair": pk,
        "exact_dup_rate_test_pct": round(_exact_dup_rate(train_src, test_src), 2),
        "near_dup_rate_test_pct": round(_near_dup_rate(train_src, test_src), 2),
        "short_seg_le5_tok_pct": round(_short_seg_pct(all_src), 2),
        "n_train": len(train_src), "n_val": len(val_src), "n_test": len(test_src),
    })

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(f"{RESULTS_DIR}/table_dataset_audit.csv", index=False)
print("Dataset audit (contamination / near-duplicate overlap / short segments)")
display(audit_df)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
proxy_plot = proxy_summary.reset_index()
sns.barplot(data=proxy_plot, x="model", y="oov_proxy_pct", hue="model", legend=False,
            order=MODEL_NAMES, ax=axes[0], palette="viridis")
axes[0].set_title("OOV proxy rate by model")
axes[0].set_ylabel("OOV rate (%)")
axes[0].tick_params(axis="x", rotation=20)

sns.barplot(data=proxy_plot, x="model", y="length_disfluency_proxy", hue="model", legend=False,
            order=MODEL_NAMES, ax=axes[1], palette="viridis")
axes[1].set_title("Length-disfluency proxy by model")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig7_error_proxy.png", dpi=150)
plt.show()


## 21B. Radar chart, per-language gain, consolidated dashboard

In [ ]:
# multi-dimensional radar comparison across all models
from math import pi

radar_data = {pk: table3[pk] for pk in LANG_PAIRS}
radar_data["Avg BLEU"] = table3["Avg"]
compute_col = "CPU hrs" if "CPU hrs" in table3.columns else "GPU hrs"
radar_data["Efficiency (1/hrs)"] = 1.0 / table3[compute_col]
radar_df = pd.DataFrame(radar_data)

radar_norm = radar_df.copy()
for col in radar_norm.columns:
    lo, hi = radar_norm[col].min(), radar_norm[col].max()
    radar_norm[col] = 10 * (radar_norm[col] - lo) / (hi - lo) if hi > lo else 5.0

labels = list(radar_norm.columns)
n = len(labels)
angles = [i / float(n) * 2 * pi for i in range(n)]
angles += angles[:1]

plt.figure(figsize=(7, 7))
ax = plt.subplot(111, polar=True)
for model in MODEL_NAMES:
    values = radar_norm.loc[model].tolist()
    values += values[:1]
    ax.plot(angles, values, label=model)
    ax.fill(angles, values, alpha=0.08)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=8)
plt.title("Multi-dimensional model comparison (radar)")
plt.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=8)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig8_radar.png", dpi=150)
plt.show()


In [ ]:
# BLEU gain over the LSTM-Attention baseline, by language pair
baseline_row = table3.loc["LSTM-Attention"]
gain_rows = []
for model in MODEL_NAMES:
    if model == "LSTM-Attention":
        continue
    for pk in LANG_PAIRS:
        gain_rows.append({"model": model, "pair": pk, "gain": table3.loc[model, pk] - baseline_row[pk]})
gain_df = pd.DataFrame(gain_rows)

plt.figure(figsize=(8, 5))
sns.barplot(data=gain_df, x="pair", y="gain", hue="model")
plt.axhline(0, color="black", linewidth=0.8)
plt.ylabel("BLEU gain over LSTM-Attention baseline")
plt.title("BLEU gain over baseline, by language pair")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig9_gain_over_baseline.png", dpi=150)
plt.show()


In [ ]:
# consolidated dashboard summarising the above figures in one view
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

table3["Avg"].reindex(MODEL_NAMES).plot(kind="bar", ax=axes[0, 0], color="steelblue")
axes[0, 0].set_title("(a) Average BLEU"); axes[0, 0].tick_params(axis="x", rotation=20)

sns.barplot(data=plot_df_melt, x="pair", y="BLEU", hue="model", ax=axes[0, 1], hue_order=MODEL_NAMES)
axes[0, 1].set_title("(b) BLEU by language pair"); axes[0, 1].legend(fontsize=7)

compute_col = "CPU hrs" if "CPU hrs" in table3.columns else "GPU hrs"
axes[0, 2].scatter(table3[compute_col], table3["Avg"], s=60)
for m in MODEL_NAMES:
    axes[0, 2].annotate(m, (table3.loc[m, compute_col], table3.loc[m, "Avg"]), fontsize=7)
axes[0, 2].set_xlabel(compute_col); axes[0, 2].set_ylabel("Avg BLEU"); axes[0, 2].set_title("(c) Quality vs. cost")

sns.heatmap(heat_data, annot=True, fmt=".1f", cmap="viridis", ax=axes[1, 0])
axes[1, 0].set_title("(d) BLEU heatmap")

_first_pair = next(iter(LANG_PAIRS))
for r in results:
    if r["pair"] == _first_pair and r["seed"] == SEEDS[0] and r["history"].get("train_loss"):
        h = r["history"]
        xs = h["step"] if h.get("step") else list(range(1, len(h["train_loss"]) + 1))
        axes[1, 1].plot(xs, h["train_loss"], label=r["model"])
axes[1, 1].set_title("(e) Training loss"); axes[1, 1].set_xlabel("Step"); axes[1, 1].legend(fontsize=7)

sns.boxplot(data=df, x="model", y="test_bleu", order=MODEL_NAMES, ax=axes[1, 2])
sns.stripplot(data=df, x="model", y="test_bleu", order=MODEL_NAMES, ax=axes[1, 2], color="black", alpha=0.5)
axes[1, 2].set_title("(f) BLEU distribution"); axes[1, 2].tick_params(axis="x", rotation=20)

plt.suptitle("Consolidated experimental dashboard", fontsize=14)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig11_dashboard.png", dpi=150)
plt.show()


## 22. Save results and figures



In [ ]:
with open(f"{RESULTS_DIR}/hypotheses_sample.json", "w") as f:
    sample = {f"{r['model']}__{r['pair']}__seed{r['seed']}": r["hyps"][:10] for r in results}
    json.dump(sample, f, indent=2, ensure_ascii=False)

print("All results, tables and figures saved under:", RESULTS_DIR)
print(os.listdir(RESULTS_DIR))